In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
trades = pd.read_csv('historical_data.csv')
sentiment = pd.read_csv('fear_greed_index.csv')

print("Trades Shape:", trades.shape)
print("Sentiment Shape:", sentiment.shape)

Trades Shape: (211224, 16)
Sentiment Shape: (2644, 4)


In [3]:
print(trades.columns)
print(sentiment.columns)

Index(['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side',
       'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL',
       'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID',
       'Timestamp'],
      dtype='object')
Index(['timestamp', 'value', 'classification', 'date'], dtype='object')


In [4]:
trades.columns = trades.columns.str.strip().str.lower().str.replace(' ', '_')
sentiment.columns = sentiment.columns.str.strip().str.lower().str.replace(' ', '_')

print(trades.columns)
print(sentiment.columns)

Index(['account', 'coin', 'execution_price', 'size_tokens', 'size_usd', 'side',
       'timestamp_ist', 'start_position', 'direction', 'closed_pnl',
       'transaction_hash', 'order_id', 'crossed', 'fee', 'trade_id',
       'timestamp'],
      dtype='object')
Index(['timestamp', 'value', 'classification', 'date'], dtype='object')


In [5]:
print("Trades date sample:")
print(trades['timestamp'].head())

print("\nSentiment date sample:")
print(sentiment.head())

Trades date sample:
0    1.730000e+12
1    1.730000e+12
2    1.730000e+12
3    1.730000e+12
4    1.730000e+12
Name: timestamp, dtype: float64

Sentiment date sample:
    timestamp  value classification        date
0  1517463000     30           Fear  2018-02-01
1  1517549400     15   Extreme Fear  2018-02-02
2  1517635800     40           Fear  2018-02-03
3  1517722200     24   Extreme Fear  2018-02-04
4  1517808600     11   Extreme Fear  2018-02-05


In [6]:
# Convert trades timestamp → only DATE
trades['timestamp'] = pd.to_datetime(trades['timestamp'], errors='coerce')
print(trades['timestamp'])
trades['date'] = trades['timestamp'].dt.date
print(trades['date'])

# Convert sentiment date properly
sentiment['date'] = pd.to_datetime(sentiment['date'], errors='coerce')
print(sentiment['date'])
sentiment['date'] = sentiment['date'].dt.date
print(sentiment['date'] )

0        1970-01-01 00:28:50
1        1970-01-01 00:28:50
2        1970-01-01 00:28:50
3        1970-01-01 00:28:50
4        1970-01-01 00:28:50
                 ...        
211219   1970-01-01 00:29:10
211220   1970-01-01 00:29:10
211221   1970-01-01 00:29:10
211222   1970-01-01 00:29:10
211223   1970-01-01 00:29:10
Name: timestamp, Length: 211224, dtype: datetime64[ns]
0         1970-01-01
1         1970-01-01
2         1970-01-01
3         1970-01-01
4         1970-01-01
             ...    
211219    1970-01-01
211220    1970-01-01
211221    1970-01-01
211222    1970-01-01
211223    1970-01-01
Name: date, Length: 211224, dtype: object
0      2018-02-01
1      2018-02-02
2      2018-02-03
3      2018-02-04
4      2018-02-05
          ...    
2639   2025-04-28
2640   2025-04-29
2641   2025-04-30
2642   2025-05-01
2643   2025-05-02
Name: date, Length: 2644, dtype: datetime64[ns]
0       2018-02-01
1       2018-02-02
2       2018-02-03
3       2018-02-04
4       2018-02-05
           .

In [7]:
print("Trades dates range:", trades['date'].min(), "to", trades['date'].max())
print("Sentiment dates range:", sentiment['date'].min(), "to", sentiment['date'].max())

Trades dates range: 1970-01-01 to 1970-01-01
Sentiment dates range: 2018-02-01 to 2025-05-02


In [8]:
common_dates = set(trades['date']).intersection(set(sentiment['date']))
print("Common dates count:", len(common_dates))

Common dates count: 0


In [9]:
# Sort both datasets
trades = trades.sort_values('timestamp')
sentiment['date_time'] = pd.to_datetime(sentiment['date'])
sentiment = sentiment.sort_values('date_time')

# Merge using nearest date
df = pd.merge_asof(
    trades,
    sentiment,
    left_on='timestamp',
    right_on='date_time',
    direction='nearest'
)

# Rename
df.rename(columns={'classification': 'sentiment'}, inplace=True)

In [10]:
print(df['sentiment'].value_counts(dropna=False))

sentiment
Fear    211224
Name: count, dtype: int64


In [11]:
trade_counts = df['account'].value_counts()
threshold = trade_counts.median()

df['trader_type'] = df['account'].apply(
    lambda x: 'Frequent' if trade_counts[x] > threshold else 'Infrequent'
)

df[['account', 'trader_type']].head()

,account,trader_type
0,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,Infrequent
1,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,Infrequent
2,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,Infrequent
3,0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23,Frequent
4,0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23,Frequent


In [11]:
trade_counts = df['account'].value_counts()
threshold = trade_counts.median()

df['trader_type'] = df['account'].apply(
    lambda x: 'Frequent' if trade_counts[x] > threshold else 'Infrequent'
)

df[['account', 'trader_type']].head()

,account,trader_type
0,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,Infrequent
1,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,Infrequent
2,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,Infrequent
3,0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23,Frequent
4,0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23,Frequent


In [14]:
# Create win column
df['win'] = df['closed_pnl'] > 0

In [ ]:
win_rate = df.groupby('sentiment')['win'].mean()
print(win_rate)

In [12]:
print("Total Trades:", len(df))
print("Average PnL:", df['closed_pnl'].mean())
print("Win Rate:", df['win'].mean())

Total Trades: 211224
Average PnL: 48.74900079269401


KeyError: 'win'

In [ ]:
import os
os.makedirs("images", exist_ok=True)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure()
sns.boxplot(x='sentiment', y='closed_pnl', data=df)
plt.title("PnL vs Sentiment")

plt.savefig("images/pnl_vs_sentiment.png")  # ✅ SAVE
plt.show()

In [ ]:
win_rate = df.groupby('sentiment')['win'].mean().reset_index()

plt.figure()
sns.barplot(x='sentiment', y='win', data=win_rate)
plt.title("Win Rate by Sentiment")

plt.savefig("images/winrate.png")  # ✅ SAVE
plt.show()

In [ ]:
plt.figure()
sns.boxplot(x='sentiment', y='size_usd', data=df)
plt.title("Trade Size by Sentiment")

plt.savefig("images/trade_size.png")  # ✅ SAVE
plt.show()